In [ ]:
# Pipeline motor imagery (run 4, 8, 12): sinistra vs destra immaginata.
# Validazione Leave-One-Run-Out: a turno una run fa da test e le altre due da training.
# --- Preambolo standard ------------------------------------------------------
# Risale l'albero fino alla root del progetto e rende importabili i moduli in src/util,
# cosi' il notebook funziona indipendentemente dalla cartella da cui parte il kernel.
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "dataset_description.json").exists()
)
DATA  = PROJECT_ROOT / "data"
PLOTS = PROJECT_ROOT / "src" / "plots"
sys.path.insert(0, str(PROJECT_ROOT / "src"))
# -----------------------------------------------------------------------------

import time
import warnings
import numpy as np
import mne
from mne.decoding import CSP
from mne_bids import BIDSPath, read_raw_bids
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.exceptions import ConvergenceWarning
from util.preprocessing import create_sliding_windows, create_window_labels
from util.channel_selection import select_discriminative_channels, IMAGERY_REST_ACTIVE

warnings.filterwarnings("ignore", category=ConvergenceWarning)
mne.set_log_level('WARNING')

# --- Configurazione ----------------------------------------------------------

root = DATA
runs = ["4", "8", "12"]                       # Le run che vengono prese in considerazione
test_run_order = [runs[2], runs[1], runs[0]]  # Ordine in cui vengono usate le run come test set
first_person = 1
people = 30

window_size = 2         # Lunghezza finestra in secondi
step_size = 0.5         # Lunghezza passo in secondi
label_threshold = 0.8   # Frazione minima di campioni di una classe perche' la finestra le venga assegnata

# Selezione dei canali:
#   "auto"  -> ricalcolata dentro ogni fold sulle sole run di training (nessun leakage)
#   "fixed" -> lista fissa qui sotto, comoda per le prove veloci
CHANNEL_MODE = "auto"
FIXED_CHANNELS = ["C1", "C2"]
N_CHANNELS = 10

# Soglia probabilistica: le predizioni sotto soglia vengono scartate invece di essere emesse.
# Parte alta e scende finche' non resta accettata almeno MIN_ACCEPTED_RATIO del test set,
# per cui l'accuratezza va sempre letta insieme alla percentuale di campioni scartati.
START_THRESHOLD = 0.90
MIN_THRESHOLD = 0.50
MIN_ACCEPTED_RATIO = 0.70

all_accuracy = np.zeros((people, len(test_run_order)))   # Accuratezza per soggetto e per run di test
all_discarded = np.zeros((people, len(test_run_order)))  # Percentuale di campioni scartati
cm_sum = np.zeros((2, 2))

# --- Loop principale ---------------------------------------------------------

# Eseguiamo la scansione di tutti i soggetti
for i in range(first_person, first_person + people):
    subject = f"{i:03d}"
    subject_start = time.time()

    print("=" * 60)
    print(f"Paziente {subject}")
    print("=" * 60)

    for test_index, test_run in enumerate(test_run_order):
        test_start = time.time()
        train_runs = [run for run in runs if run != test_run]

        # I canali vengono scelti guardando SOLO le run di training di questo fold:
        # includere la run di test farebbe entrare informazione del test nella selezione.
        if CHANNEL_MODE == "auto":
            channels_of_interest = select_discriminative_channels(
                subject,
                train_runs,
                root,
                class_map=IMAGERY_REST_ACTIVE,
                n_channels=N_CHANNELS,
            )
        else:
            channels_of_interest = FIXED_CHANNELS

        X_train = []
        y_train = []
        groups_train = []   # trial di provenienza di ogni finestra di training

        X_test = []
        y_test = []

        trial_offset = 0    # rende unici gli id dei trial tra run diverse

        # Per ogni soggetto eseguiamo la scansione sulle run di nostro interesse
        for run in runs:
            bids_path = BIDSPath(   # Specifichiamo il percorso del dataset e BIDS eseguira' correttamente la scansione
                subject=subject,
                task="motion",
                run=run,
                datatype="eeg",
                root=root,
            )

            try:
                # Fase 1: lettura dei dati
                raw = read_raw_bids(bids_path, verbose=False)
                events, event_id = mne.events_from_annotations(raw, verbose=False)
                raw.load_data(verbose=False)  # Carico i dati in memoria per poter filtrare ecc.
                raw.filter(l_freq=8, h_freq=30, verbose=False)  # Filtro passa banda 8-30 Hz
                raw.set_eeg_reference('average', projection=False, verbose=False)  # Riferimento medio

                event_map = {
                    event_id['TASK2T0']: 1,
                    event_id['TASK2T1']: 2,
                    event_id['TASK2T2']: 3
                }

                # Fase 2: pre-processing
                windows, window_samples, step_samples, total_samples = create_sliding_windows(
                    raw,
                    window_size,
                    step_size
                )

                picks = mne.pick_channels(raw.ch_names, channels_of_interest)
                windows = windows[:, picks, :]

                y, groups = create_window_labels(
                    events,
                    event_map,
                    total_samples,
                    window_samples,
                    step_samples,
                    threshold=label_threshold,
                    return_groups=True
                )

                # Due trial di run diverse non devono finire nello stesso gruppo.
                # Il sentinella -1 (finestre fuori da ogni trial) va lasciato com'e'.
                groups = np.where(groups >= 0, groups + trial_offset, -1)
                trial_offset += len(events)

                # Divido la classificazione in due step: prima distinguo tra stato di riposo e di attivazione.
                # In caso di attivazione, distinguo tra sinistra e destra. Qui faccio la seconda parte.
                y_rest_active = np.where(y == 1, 0, 1)

                # Variabile per addestrare il modello sul sinistra/destra
                mask_active = y != 1
                y_active = y[mask_active]
                y_lr = np.where(y_active == 2, 0, 1)

                # 1) Assegno a y_binary le etichette desiderate e 2) se voglio rest/active commento la seconda riga
                y_binary = y_lr
                windows = windows[mask_active]
                groups_binary = groups[mask_active]

                if run in train_runs:
                    X_train.append(windows)
                    y_train.append(y_binary)
                    groups_train.append(groups_binary)
                else:
                    X_test.append(windows)
                    y_test.append(y_binary)

            except Exception as e:
                print(f"Errore {subject} run {run}: {e}")

        if not X_train or not X_test:
            print(f"  Dati insufficienti per la run di test {test_run}")
            continue

        X_train = np.concatenate(X_train, axis=0)
        y_train = np.concatenate(y_train)
        groups_train = np.concatenate(groups_train)

        X_test = np.concatenate(X_test, axis=0)
        y_test = np.concatenate(y_test)

        # Fase 3: feature extraction e classificazione
        pipe = Pipeline([
            ("csp", CSP(reg='ledoit_wolf')),  # reg aiuta la stabilita' con finestre corte
            ("scaler", StandardScaler()),     # Fondamentale per SVM
            ("svm", SVC(probability=True, class_weight='balanced'))
        ])

        param_grid = {
            "csp__n_components": [2, 4, 6, 8],
            "csp__log": [True, False],
            "svm__C": [0.1, 1, 10, 100],
            "svm__gamma": ["scale", "auto"],
            "svm__kernel": ["rbf", "linear"]
        }

        # Le finestre si sovrappongono al 75%: se la cross validation interna le dividesse a
        # caso, finestre quasi identiche finirebbero sia in train sia in validation e lo score
        # sarebbe gonfiato. Raggruppando per trial restano tutte dalla stessa parte dello split.
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

        grid = GridSearchCV(
            pipe,
            param_grid,
            cv=cv,
            scoring="balanced_accuracy",
            n_jobs=-1
        )

        grid.fit(X_train, y_train, groups=groups_train)

        # Fase 4: predizione con logica probabilistica
        probs = grid.predict_proba(X_test)
        max_probs = np.max(probs, axis=1)
        predictions = np.argmax(probs, axis=1)

        # Scelta dinamica del threshold (valore di probabilita' sotto il quale la classificazione e' scartata)
        threshold = START_THRESHOLD
        accepted_mask = max_probs >= threshold

        while np.sum(accepted_mask) < MIN_ACCEPTED_RATIO * len(y_test) and threshold > MIN_THRESHOLD:
            threshold -= 0.05
            accepted_mask = max_probs >= threshold

        accepted_predictions = predictions[accepted_mask]
        accepted_true_labels = y_test[accepted_mask]

        accepted = int(np.sum(accepted_mask))
        total = len(y_test)
        discarded = total - accepted

        # Balanced misura un'accuratezza media tra le classi (cosi' se ho 70% su una classe
        # e 0 sull'altra ottengo comunque un valore adeguato)
        accuracy = balanced_accuracy_score(accepted_true_labels, accepted_predictions)

        all_accuracy[i - first_person, test_index] = accuracy
        all_discarded[i - first_person, test_index] = 100 * discarded / total
        cm_sum += confusion_matrix(accepted_true_labels, accepted_predictions)

        # Stampa dei risultati della singola run: accuratezza e scarti vanno sempre letti in coppia
        test_elapsed = time.time() - test_start
        print(f"  Run test: {test_run} | Threshold: {threshold:.2f} | Canali: {channels_of_interest}")
        print(f"  Campioni: {total} tot / {accepted} accettati / {discarded} scartati ({100*discarded/total:.1f}%)")
        print(f"  Balanced accuracy sugli accettati: {accuracy*100:.2f}% | "
              f"Best params: C={grid.best_params_['svm__C']}, components={grid.best_params_['csp__n_components']}")
        print(f"  Tempo: {test_elapsed:.1f}s\n")

    # Stampa dei risltati medi per soggetto
    patient_mean = all_accuracy[i - first_person].mean()
    patient_discarded = all_discarded[i - first_person].mean()
    print(f"  Paziente {subject}: {patient_mean*100:.2f}% con {patient_discarded:.1f}% di scarti "
          f"| Tempo: {time.time() - subject_start:.1f}s\n")

# --- Riepilogo ---------------------------------------------------------------

n_folds = all_accuracy.size
print("=" * 60)
print(f"Accuratezza media:      {all_accuracy.mean()*100:.2f}%")
print(f"Deviazione standard:    {all_accuracy.std()*100:.2f}%")
print(f"Campioni scartati:      {all_discarded.mean():.1f}% in media")
print(f"Matrice di confusione media:\n{(cm_sum / n_folds).astype(int)}")


Paziente 001
  Run test: 12 | Threshold: 0.75 | Canali: ['C3', 'Cp3', 'Fc3', 'Cp1', 'C5', 'Cp5', 'Cp6', 'C1', 'Fc4', 'Fc1']
  Campioni: 84 tot / 59 accettati / 25 scartati (29.8%)
  Balanced accuracy sugli accettati: 79.82% | Best params: C=0.1, components=8
  Tempo: 28.1s

  Run test: 8 | Threshold: 0.70 | Canali: ['C3', 'Fc3', 'Fc4', 'C5', 'C4', 'Fc2', 'Fc1', 'Fc5', 'Cp6', 'Fc6']
  Campioni: 84 tot / 62 accettati / 22 scartati (26.2%)
  Balanced accuracy sugli accettati: 65.62% | Best params: C=1, components=8
  Tempo: 15.5s

  Run test: 4 | Threshold: 0.60 | Canali: ['Fc4', 'Fc3', 'C3', 'Fc2', 'Fc1', 'Fcz', 'C5', 'C2', 'C4', 'Fc6']
  Campioni: 84 tot / 63 accettati / 21 scartati (25.0%)
  Balanced accuracy sugli accettati: 66.08% | Best params: C=1, components=2
  Tempo: 15.1s

  Paziente 001: 70.51% con 27.0% di scarti | Tempo: 58.6s

Paziente 002
  Run test: 12 | Threshold: 0.80 | Canali: ['C5', 'Fc5', 'Cp3', 'Cp6', 'Cp1', 'C1', 'Cz', 'C3', 'Fcz', 'C6']
  Campioni: 84 tot / 60 acc